# 06｜选择模型、校准、评测与批量预测

130U-FinLLM｜130U 金融大语言模型

先比较 SFT 与 DPO 的开发集结果，固定最终检查点，再分别使用温度拟合集与阈值选择集。最后才打开 test。分数是三类立场概率的方向期望，不是收益率或价格上涨概率。

每个 Notebook 都包含独立环境入口。Colab 菜单：运行时 → 更改运行时类型 → T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
command(sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<3", *tokenizer_requirements, "scikit-learn>=1.5,<2")
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps")
check_tokenizer_compatibility(tokenizer_requirements)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
# 可选持久化：需要你在 Google 的授权界面完成登录。
MOUNT_DRIVE = False  # 需要备份 / 恢复时设为 True，再执行此单元格。
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive 项目目录：", DRIVE_ROOT)
else:
    print("尚未挂载 Drive；/content 是临时磁盘。正式训练前请启用备份。")

In [ ]:
# 新 Notebook 可能分配新的虚拟机；按需恢复前一阶段的文件。
RESTORE_FROM_DRIVE = False
LIVE_STAGES_TO_RESTORE = []  # 中断恢复时填写，例如 ["pretrain"]；会覆盖本地对应阶段。
if RESTORE_FROM_DRIVE:
    if not MOUNT_DRIVE:
        raise RuntimeError("请先把 MOUNT_DRIVE 设为 True 并完成授权。")
    for folder in ("data", "tokenizer", "runs"):
        source = DRIVE_ROOT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", PROJECT / folder, "--working-copy")
    for stage in LIVE_STAGES_TO_RESTORE:
        if stage not in ("pretrain", "domain", "sft", "dpo"):
            raise ValueError("未知检查点阶段：" + str(stage))
        source = DRIVE_ROOT / "live_checkpoints" / stage
        if not source.exists():
            raise FileNotFoundError("缺少阶段自动备份：" + str(source))
        finllm("sync", "--source", source, "--destination", PROJECT / "runs" / stage, "--working-copy")
    print("恢复完成。自动检查点只用于匹配原配置、tokenizer 和数据的继续训练；请指定 RESUME_CHECKPOINT。")

In [ ]:
require("tokenizer/english/tokenizer.json", "data/task/development.jsonl")
for stage in ("sft", "dpo"):
    checkpoint = f"runs/{stage}/best.pt"
    require(checkpoint)
    finllm("evaluate", "--checkpoint", checkpoint, "--tokenizer", "tokenizer/english", "--input", "data/task/development.jsonl", "--output", f"runs/evaluation/{stage}_development.json")
    print(stage, Path(f"runs/evaluation/{stage}_development.json").read_text())

下面的 FINAL_CHECKPOINT 必须根据上面的开发集结果选择。校准文件绑定模型和 tokenizer 的 SHA256，更换模型后必须重新校准。不要根据 test 结果回头选择 SFT 或 DPO。

In [ ]:
FINAL_CHECKPOINT = "runs/sft/best.pt"  # 按开发集结果选择。
require(FINAL_CHECKPOINT, "data/task/calibration_temperature.jsonl", "data/task/calibration_threshold.jsonl")
finllm("calibrate", "--checkpoint", FINAL_CHECKPOINT, "--tokenizer", "tokenizer/english", "--temperature-data", "data/task/calibration_temperature.jsonl", "--threshold-data", "data/task/calibration_threshold.jsonl", "--output", "runs/evaluation/calibration.json")

In [ ]:
require("data/task/test.jsonl", "runs/evaluation/calibration.json")
finllm("evaluate", "--checkpoint", FINAL_CHECKPOINT, "--tokenizer", "tokenizer/english", "--input", "data/task/test.jsonl", "--calibration", "runs/evaluation/calibration.json", "--output", "runs/evaluation/test.json")
print(Path("runs/evaluation/test.json").read_text())

批量预测输入为 JSONL，每行含 text、target，可选 language='en' 与经核查的 target_aliases。需要 cashtag 或明确名称关联；普通英文单词 CAT 不自动当作股票。超长、明显不支持的语言或目标关联不足的输入返回空分数并要求复核。

In [ ]:
PREDICTION_INPUT = "data/raw/predict.jsonl"
require(PREDICTION_INPUT)
finllm("predict", "--checkpoint", FINAL_CHECKPOINT, "--tokenizer", "tokenizer/english", "--input", PREDICTION_INPUT, "--calibration", "runs/evaluation/calibration.json", "--output", "runs/predictions.jsonl")
print("已写出：runs/predictions.jsonl")

In [ ]:
if MOUNT_DRIVE:
    for folder in ("data", "tokenizer", "runs"):
        source = PROJECT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", DRIVE_ROOT / folder)
    shutil.copy2(PROJECT / "runtime_revision.txt", DRIVE_ROOT / "runtime_revision.txt")
    print("备份完成。请保留校验清单，下一运行时按清单恢复。")
else:
    print("成果当前仅在临时运行时中；请挂载 Drive 后重新运行备份单元格。")